In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"      # must be set before torch is imported
os.chdir("/home/eric/TurboVLA")              # checkpoints store repo-relative paths

import warnings; warnings.filterwarnings("ignore")
import cv2, imageio, torch, numpy as np
import ipywidgets as widgets
from IPython.display import display, Video

from mikasa_robo_suite.vla.benchmarking import (
    START_SEED, BenchmarkConfig, make_benchmark_env, select_benchmark_tasks,
    _compose_frame, _ensure_hwc_rgb, _first_scalar,
)

In [ ]:
# class RandomPolicy:
#     def start_task(self, instruction): pass
#     def reset(self): pass
#     def forward(self, obs): return torch.rand(7) * 2 - 1

# policy = RandomPolicy()

OPEN_LOOP = 4  # actions executed per predicted chunk before re-planning (1..chunk size)

from turbovla.evaluation.mikasa_policy import TurboVLAMikasaPolicy
policy = TurboVLAMikasaPolicy(
    ckpt_path="/home/eric/usrdata/llm_weights/TurboVLA/mikasa5_hist_obs_k3s1/turbovla_step_36000.pth",
    precision="bf16",
    history_eval_mode="normal",   # or "current" / "shuffled"
    num_open_loop_steps=OPEN_LOOP,
)

_orig_predict = type(policy).predict_chunk

def predict_logged(self):
    chunk = _orig_predict(self)
    print(f"step {len(self._frames) - 1:3d}: model call -> chunk {chunk.shape}, "
          f"executing {self.num_open_loop_steps}")
    return chunk

policy.predict_chunk = predict_logged.__get__(policy)


In [ ]:
def run_live(env_id, seed=START_SEED, fps=20, save_path=None, open_loop=None):
    if open_loop is not None:
        assert 1 <= open_loop <= policy.model_chunk_size, f"open_loop must be in [1, {policy.model_chunk_size}]"
        policy.num_open_loop_steps = open_loop   # read by policy.forward() each time it re-plans

    task = select_benchmark_tasks(env_ids=[env_id])[0]
    env = make_benchmark_env(env_id, BenchmarkConfig(save_videos=True))  # rgb_array render + overlays

    def frame(obs):
        rgb = obs["rgb"][0].cpu().numpy()          # (128,128,6): base | wrist
        return _compose_frame(_ensure_hwc_rgb(env.render()), rgb[..., :3], rgb[..., 3:6])

    img = widgets.Image(format="jpeg", width=768/4*3)
    label = widgets.HTML()
    display(widgets.VBox([label, img]))

    def show(f, text):
        img.value = cv2.imencode(".jpg", f[..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes()
        label.value = text

    policy.start_task(task.language_instruction)   # also resets the policy's history and action queue
    obs, _ = env.reset(seed=seed)
    frames = [frame(obs)]
    success, ret = False, 0.0
    loop_info = f"open-loop {policy.num_open_loop_steps}/{policy.model_chunk_size}"
    show(frames[-1], f"<b>{env_id}</b> seed={seed} | {loop_info}<br>{task.language_instruction}")

    try:
        for t in range(env.max_episode_steps):
            # one action per env step; the policy re-plans internally every num_open_loop_steps steps
            action = torch.as_tensor(policy.forward(obs), dtype=torch.float32).reshape(1, -1)
            obs, reward, terminated, truncated, info = env.step(action.to(env.unwrapped.device))
            ret += _first_scalar(reward, default=0.0)
            success |= bool(_first_scalar(info.get("success"), default=False))
            frames.append(frame(obs))
            show(frames[-1], f"<b>{env_id}</b> seed={seed} | {loop_info} | step {t + 1}/{env.max_episode_steps} | "
                             f"return {ret:.2f} | success {'✅' if success else '—'}<br>{task.language_instruction}")
            if _first_scalar(terminated, default=False) or _first_scalar(truncated, default=False):
                break
    finally:
        env.close()

    print(f"{env_id} seed={seed} {loop_info}: success={success} steps={t + 1} return={ret:.3f}")
    if save_path:
        imageio.mimwrite(save_path, frames, fps=fps, codec="libx264", macro_block_size=1)
    return success, frames


In [ ]:
success, frames = run_live("InterceptMedium-VLA-v0", seed=START_SEED+7, save_path="/tmp/ep.mp4", open_loop=OPEN_LOOP)

In [ ]:
Video("/tmp/ep.mp4", embed=True, html_attributes="controls loop autoplay")

In [ ]:
# TASKS = ["ShellGamePush-VLA-v0", "InterceptMedium-VLA-v0", "RememberColor5-VLA-v0",
#          "TakeItBack-VLA-v0", "RememberShapeAndColor3x3-VLA-v0"]
# results = {t: [run_live(t, seed=START_SEED + i, open_loop=OPEN_LOOP)[0] for i in range(3)] for t in TASKS}
# {t: np.mean(s) for t, s in results.items()}


In [ ]:
class Teleop:
    """Click a button to run one env step with that action. 'Save' writes the video."""
    def __init__(self, env_id, seed=START_SEED, step=0.3, rot_step=0.3):
        self.env_id, self.seed = env_id, seed
        self.task = select_benchmark_tasks(env_ids=[env_id])[0]
        self.env = make_benchmark_env(env_id, BenchmarkConfig(save_videos=True))
        self.obs, _ = self.env.reset(seed=seed)
        self.gripper, self.t, self.ret, self.success, self.done = 1.0, 0, 0.0, False, False
        self.frames = [self._frame()]
        self.proprios = [self._proprio()]          # add: one row per env step, (T+1, 7)
        self.actions = []

        self.img, self.label = widgets.Image(format="jpeg", width=768/4*3), widgets.HTML()
        self.step_sl = widgets.FloatSlider(value=step, min=0.05, max=1.0, step=0.05, description="pos step")
        self.rot_sl = widgets.FloatSlider(value=rot_step, min=0.05, max=1.0, step=0.05, description="rot step")
        self.raw = widgets.Text(placeholder="raw: 0 0 0 0 0 0 1", description="action")

        def btn(text, fn):
            b = widgets.Button(description=text, layout=widgets.Layout(width="70px"))
            b.on_click(lambda _: fn()); return b
        def move(i, sign, rot=False):
            a = np.zeros(7); a[i] = sign * (self.rot_sl.value if rot else self.step_sl.value)
            return lambda: self.act(a)

        pos = widgets.HBox([btn(f"{s}{ax}", move(i, 1 if s == "+" else -1))
                            for i, ax in enumerate("xyz") for s in "+-"])
        rot = widgets.HBox([btn(f"{s}{ax}", move(i + 3, 1 if s == "+" else -1, rot=True))
                            for i, ax in enumerate(["rx", "ry", "rz"]) for s in "+-"])
        misc = widgets.HBox([btn("grip", self.toggle_grip), btn("noop", lambda: self.act(np.zeros(7))),
                             btn("send", self.send_raw), btn("save", self.save)])
        display(widgets.VBox([self.label, self.img, self.step_sl, self.rot_sl, pos, rot, misc, self.raw]))
        self._show()

    def _frame(self):
        rgb = self.obs["rgb"][0].cpu().numpy()
        return _compose_frame(_ensure_hwc_rgb(self.env.render()), rgb[..., :3], rgb[..., 3:6])

    def _proprio(self):                            
        return self.obs["proprio"][0].cpu().numpy().astype(np.float32).reshape(-1)

    def _show(self):
        self.img.value = cv2.imencode(".jpg", self.frames[-1][..., ::-1], [cv2.IMWRITE_JPEG_QUALITY, 85])[1].tobytes()
        p = self.obs["proprio"].reshape(-1)[:7].cpu().numpy()   # xyz(3) + rpy(3) + gripper(1)
        self.label.value = (f"<b>{self.env_id}</b> seed={self.seed} | step {self.t}/{self.env.max_episode_steps} | "
                            f"gripper {'open' if self.gripper > 0 else 'closed'} | return {self.ret:.2f} | "
                            f"success {'✅' if self.success else '—'}{' | DONE' if self.done else ''}"
                            f"<br>xyz=({p[0]:+.3f}, {p[1]:+.3f}, {p[2]:+.3f}) "
                            f"rpy=({p[3]:+.2f}, {p[4]:+.2f}, {p[5]:+.2f}) grip={p[6]:+.3f}"
                            f"<br>{self.task.language_instruction}")

    def act(self, a):
        a = np.asarray(a, dtype=np.float32).copy()
        a[6] = self.gripper
        action = torch.as_tensor(np.clip(a, -1, 1)).reshape(1, -1).to(self.env.unwrapped.device)
        self.obs, reward, terminated, truncated, info = self.env.step(action)
        self.t += 1
        self.ret += _first_scalar(reward, default=0.0)
        self.success |= bool(_first_scalar(info.get("success"), default=False))
        self.done |= bool(_first_scalar(terminated, default=False) or _first_scalar(truncated, default=False))
        self.frames.append(self._frame()); self._show()

    def toggle_grip(self):
        self.gripper = -self.gripper
        self.act(np.zeros(7))

    def history(self):                             # add: whole trajectory as arrays
        return {"proprio": np.stack(self.proprios), "actions": np.array(self.actions).reshape(-1, 7)}
        
    def send_raw(self):
        vals = [float(v) for v in self.raw.value.replace(",", " ").split()]
        assert len(vals) == 7, "need 7 values"
        self.gripper = vals[6]
        self.act(vals)

    def save(self, path="/tmp/teleop.mp4", fps=20):
        imageio.mimwrite(path, self.frames, fps=fps, codec="libx264", macro_block_size=1)
        self.label.value += f"<br>saved {path}"

    def close(self):
        self.env.close()

tele = Teleop("TakeItBack-VLA-v0", seed=0 + 7)